# **ALZ-Foresight - ADNI Images Simple Processing Pipeline**

### Model 1: First Attempt
*Highest Accuracy at 0.386 / 3 Epochs*

In [ ]:
# =========================
# 1) Setup
# =========================
!pip install pydicom

import os
import zipfile
import random
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# CONFIG (NEW)
# =========================
MAX_SAMPLES = 20000   # 🔥 adjust (10k–30k recommended)
IMG_SIZE = 96         # 🔥 faster
BATCH_SIZE = 32       # 🔥 faster


# =========================
# 2) Paths
# =========================
zip_path = "/content/drive/MyDrive/ADNI_DATA/images/MRI.zip"
diagnosis_csv_path = "Diagnositic_Summary.csv"


# =========================
# 3) Load Diagnosis CSV
# =========================
df = pd.read_csv(diagnosis_csv_path)

df = df[["PTID", "EXAMDATE", "DXNORM", "DXMCI", "DXAD"]]
df["EXAMDATE"] = pd.to_datetime(df["EXAMDATE"], errors="coerce")

df = df.sort_values("EXAMDATE")
latest_df = df.groupby("PTID").tail(1)

labels_dict = {}

for _, row in latest_df.iterrows():
    if row["DXAD"] == 1:
        labels_dict[row["PTID"]] = 2
    elif row["DXMCI"] == 1:
        labels_dict[row["PTID"]] = 1
    elif row["DXNORM"] == 1:
        labels_dict[row["PTID"]] = 0

print("Total labeled subjects:", len(labels_dict))


# =========================
# 4) Open ZIP
# =========================
zip_ref = zipfile.ZipFile(zip_path, 'r')
all_files = [f for f in zip_ref.namelist() if f.endswith(".dcm")]
print("Total DICOM files:", len(all_files))


# =========================
# 5) Extract Subject ID
# =========================
def extract_subject_id(path):
    for part in path.split("/"):
        if "_S_" in part:
            return part
    return None


# =========================
# 6) Group by SUBJECT (CRITICAL FIX)
# =========================
subject_to_files = {}

for f in all_files:
    subject_id = extract_subject_id(f)
    if subject_id in labels_dict:
        subject_to_files.setdefault(subject_id, []).append(f)

print("Total subjects with images:", len(subject_to_files))


# =========================
# 7) SUBJECT SPLIT (NO LEAKAGE)
# =========================
subjects = list(subject_to_files.keys())
random.shuffle(subjects)

split = int(0.8 * len(subjects))
train_subjects = subjects[:split]
val_subjects = subjects[split:]

train_files = []
val_files = []

for s in train_subjects:
    train_files.extend(subject_to_files[s])

for s in val_subjects:
    val_files.extend(subject_to_files[s])


# =========================
# 8) LIMIT DATA (SPEED)
# =========================
train_files = train_files[:MAX_SAMPLES]
val_files = val_files[:int(MAX_SAMPLES * 0.2)]

print("Train:", len(train_files), "Val:", len(val_files))


# =========================
# 9) Dataset Class
# =========================
class ADNIDataset(Dataset):
    def __init__(self, zip_path, file_list, labels_dict, transform=None):
        self.zip_path = zip_path
        self.file_list = file_list
        self.labels_dict = labels_dict
        self.transform = transform
        self.zip_ref = None

    def _get_zip(self):
        if self.zip_ref is None:
            self.zip_ref = zipfile.ZipFile(self.zip_path, 'r')
        return self.zip_ref

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        zip_ref = self._get_zip()
        file_path = self.file_list[idx]

        try:
            with zip_ref.open(file_path) as f:
                dcm = pydicom.dcmread(f)
                img = dcm.pixel_array.astype(np.float32)
        except:
            return self.__getitem__((idx + 1) % len(self.file_list))

        img = (img - img.min()) / (img.max() - img.min() + 1e-8)
        img = Image.fromarray((img * 255).astype(np.uint8))

        if self.transform:
            img = self.transform(img)

        subject_id = extract_subject_id(file_path)
        label = self.labels_dict[subject_id]

        return img, label


# =========================
# 10) Transforms
# =========================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


# =========================
# 11) DataLoaders
# =========================
train_dataset = ADNIDataset(zip_path, train_files, labels_dict, transform)
val_dataset = ADNIDataset(zip_path, val_files, labels_dict, transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


# =========================
# 12) Model
# =========================
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * (IMG_SIZE//8) * (IMG_SIZE//8), 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.fc(self.conv(x))


model = SimpleCNN().to(device)


# =========================
# 13) Training
# =========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def train_model(model, train_loader, val_loader, epochs=3):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs).argmax(1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        print(f"Epoch {epoch+1}")
        print("Train Loss:", total_loss / len(train_loader))
        print("Val Accuracy:", correct / total)


train_model(model, train_loader, val_loader)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Total labeled subjects: 521
Total DICOM files: 1604356
Total subjects with images: 486
Train: 20000 Val: 4000
Epoch 1
Train Loss: 0.5099616573572159
Val Accuracy: 0.32725
Epoch 2
Train Loss: 0.12709539957214147
Val Accuracy: 0.323
Epoch 3
Train Loss: 0.0684499888587743
Val Accuracy: 0.386


### Model 2: Data Augmentation
*Highest Accuracy at 0.612 / 5 Epochs*

In [ ]:
# =========================
# 1) Setup
# =========================
!pip install pydicom

import os
import zipfile
import random
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# CONFIG (UPDATED)
# =========================
MAX_SAMPLES = 20000
IMG_SIZE = 96
BATCH_SIZE = 32
SLICES_PER_SUBJECT = 20   # 🔥 NEW (critical)


# =========================
# 2) Paths
# =========================
zip_path = "/content/drive/MyDrive/ADNI_DATA/images/MRI.zip"
diagnosis_csv_path = "Diagnositic_Summary.csv"


# =========================
# 3) Load Diagnosis CSV
# =========================
df = pd.read_csv(diagnosis_csv_path)

df = df[["PTID", "EXAMDATE", "DXNORM", "DXMCI", "DXAD"]]
df["EXAMDATE"] = pd.to_datetime(df["EXAMDATE"], errors="coerce")

df = df.sort_values("EXAMDATE")
latest_df = df.groupby("PTID").tail(1)

labels_dict = {}

for _, row in latest_df.iterrows():
    if row["DXAD"] == 1:
        labels_dict[row["PTID"]] = 2
    elif row["DXMCI"] == 1:
        labels_dict[row["PTID"]] = 1
    elif row["DXNORM"] == 1:
        labels_dict[row["PTID"]] = 0

print("Total labeled subjects:", len(labels_dict))


# =========================
# 4) Open ZIP
# =========================
zip_ref = zipfile.ZipFile(zip_path, 'r')
all_files = [f for f in zip_ref.namelist() if f.endswith(".dcm")]
print("Total DICOM files:", len(all_files))


# =========================
# 5) Extract Subject ID
# =========================
def extract_subject_id(path):
    for part in path.split("/"):
        if "_S_" in part:
            return part
    return None


# =========================
# 6) Group by SUBJECT
# =========================
subject_to_files = {}

for f in all_files:
    subject_id = extract_subject_id(f)
    if subject_id in labels_dict:
        subject_to_files.setdefault(subject_id, []).append(f)

print("Total subjects with images:", len(subject_to_files))


# =========================
# 7) Slice Selection (🔥 KEY IMPROVEMENT)
# =========================
def select_middle_slices(files, num_slices=SLICES_PER_SUBJECT):
    files = sorted(files)
    mid = len(files) // 2
    return files[max(0, mid - num_slices//2): mid + num_slices//2]


# =========================
# 8) SUBJECT SPLIT
# =========================
subjects = list(subject_to_files.keys())
random.shuffle(subjects)

split = int(0.8 * len(subjects))
train_subjects = subjects[:split]
val_subjects = subjects[split:]

train_files = []
val_files = []

for s in train_subjects:
    selected = select_middle_slices(subject_to_files[s])
    train_files.extend(selected)

for s in val_subjects:
    selected = select_middle_slices(subject_to_files[s])
    val_files.extend(selected)


# =========================
# 9) LIMIT DATA
# =========================
train_files = train_files[:MAX_SAMPLES]
val_files = val_files[:int(MAX_SAMPLES * 0.2)]

print("Train:", len(train_files), "Val:", len(val_files))


# =========================
# 10) Dataset Class
# =========================
class ADNIDataset(Dataset):
    def __init__(self, zip_path, file_list, labels_dict, transform=None):
        self.zip_path = zip_path
        self.file_list = file_list
        self.labels_dict = labels_dict
        self.transform = transform
        self.zip_ref = None

    def _get_zip(self):
        if self.zip_ref is None:
            self.zip_ref = zipfile.ZipFile(self.zip_path, 'r')
        return self.zip_ref

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        zip_ref = self._get_zip()
        file_path = self.file_list[idx]

        try:
            with zip_ref.open(file_path) as f:
                dcm = pydicom.dcmread(f)
                img = dcm.pixel_array.astype(np.float32)
        except:
            return self.__getitem__((idx + 1) % len(self.file_list))

        # 🔥 Better normalization
        img = (img - np.mean(img)) / (np.std(img) + 1e-8)

        # Scale to 0–255
        img = ((img - img.min()) / (img.max() - img.min() + 1e-8))
        img = Image.fromarray((img * 255).astype(np.uint8))

        if self.transform:
            img = self.transform(img)

        subject_id = extract_subject_id(file_path)
        label = self.labels_dict[subject_id]

        return img, label


# =========================
# 11) Transforms (🔥 AUGMENTATION)
# =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])


# =========================
# 12) DataLoaders
# =========================
train_dataset = ADNIDataset(zip_path, train_files, labels_dict, train_transform)
val_dataset = ADNIDataset(zip_path, val_files, labels_dict, val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


# =========================
# 13) Model (🔥 STRONGER)
# =========================
class ImprovedCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * (IMG_SIZE//8) * (IMG_SIZE//8), 128),
            nn.ReLU(),
            nn.Linear(128, 3)
        )

    def forward(self, x):
        return self.fc(self.conv(x))


model = ImprovedCNN().to(device)


# =========================
# 14) Training
# =========================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)


def train_model(model, train_loader, val_loader, epochs=5):
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        model.eval()
        correct, total = 0, 0

        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                preds = model(imgs).argmax(1)
                total += labels.size(0)
                correct += (preds == labels).sum().item()

        print(f"\nEpoch {epoch+1}")
        print("Train Loss:", total_loss / len(train_loader))
        print("Val Accuracy:", correct / total)


train_model(model, train_loader, val_loader)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Total labeled subjects: 521
Total DICOM files: 1604356
Total subjects with images: 486
Train: 7760 Val: 1960

Epoch 1
Train Loss: 0.9952688393769441
Val Accuracy: 0.6122448979591837

Epoch 2
Train Loss: 0.9912055684215247
Val Accuracy: 0.6122448979591837

Epoch 3
Train Loss: 0.9813617458068785
Val Accuracy: 0.6122448979591837

Epoch 4
Train Loss: 0.9589924770618172
Val Accuracy: 0.5948979591836735

Epoch 5
Train Loss: 0.9135235682927041
Val Accuracy: 0.48316326530612247


### Model 3: ResNet Model
*Highest Accuracy at 0.575 / 6 Epochs (Previous Versiom)*

In [ ]:
# =========================
# 1) Setup
# =========================
!pip install pydicom

import os
import zipfile
import random
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter

from google.colab import drive
drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# CONFIG
# =========================
SEED = 42
MAX_SUBJECTS = 450          # keep runtime reasonable
IMG_SIZE = 160              # stronger than 128, still manageable
BATCH_SIZE = 8              # subject-level batch
SLICES_PER_SUBJECT = 9      # odd number for symmetric middle-slice selection
NUM_WORKERS = 2
EPOCHS = 10
LR = 3e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 3                # early stopping patience

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# =========================
# 2) Paths
# =========================
zip_path = "/content/drive/MyDrive/ADNI_DATA/images/MRI.zip"
diagnosis_csv_path = "Diagnositic_Summary.csv"
best_model_path = "/content/drive/MyDrive/best_subject_resnet18_agg.pth"


# =========================
# 3) Load Labels
# =========================
df = pd.read_csv(diagnosis_csv_path)

df = df[["PTID", "EXAMDATE", "DXNORM", "DXMCI", "DXAD"]].copy()
df["EXAMDATE"] = pd.to_datetime(df["EXAMDATE"], errors="coerce")

df = df.sort_values("EXAMDATE")
latest_df = df.groupby("PTID").tail(1)

labels_dict = {}

for _, row in latest_df.iterrows():
    if row["DXAD"] == 1:
        labels_dict[row["PTID"]] = 2
    elif row["DXMCI"] == 1:
        labels_dict[row["PTID"]] = 1
    elif row["DXNORM"] == 1:
        labels_dict[row["PTID"]] = 0

print("Total labeled subjects:", len(labels_dict))


# =========================
# 4) Read ZIP
# =========================
zip_ref = zipfile.ZipFile(zip_path, "r")
all_files = [f for f in zip_ref.namelist() if f.endswith(".dcm")]
zip_ref.close()

print("Total DICOM files:", len(all_files))


# =========================
# 5) Extract Subject ID
# =========================
def extract_subject_id(path):
    for part in path.split("/"):
        if "_S_" in part:
            return part
    return None


# =========================
# 6) Group by subject
# =========================
subject_to_files = {}

for f in all_files:
    subject_id = extract_subject_id(f)
    if subject_id in labels_dict:
        subject_to_files.setdefault(subject_id, []).append(f)

# keep only subjects with enough slices
subject_to_files = {
    sid: sorted(files)
    for sid, files in subject_to_files.items()
    if len(files) >= SLICES_PER_SUBJECT
}

print("Total subjects with images:", len(subject_to_files))


# =========================
# 7) Limit subjects
# =========================
subjects = list(subject_to_files.keys())
random.shuffle(subjects)

if len(subjects) > MAX_SUBJECTS:
    subjects = subjects[:MAX_SUBJECTS]

print("Subjects used:", len(subjects))


# =========================
# 8) Subject split
# =========================
# stratified split by subject label
label_to_subjects = {0: [], 1: [], 2: []}
for sid in subjects:
    label_to_subjects[labels_dict[sid]].append(sid)

train_subjects = []
val_subjects = []

for cls in [0, 1, 2]:
    cls_subjects = label_to_subjects[cls]
    random.shuffle(cls_subjects)
    split = int(0.8 * len(cls_subjects))
    train_subjects.extend(cls_subjects[:split])
    val_subjects.extend(cls_subjects[split:])

random.shuffle(train_subjects)
random.shuffle(val_subjects)

print("Train subjects:", len(train_subjects))
print("Val subjects:", len(val_subjects))


# =========================
# 9) Slice selection
# =========================
def select_middle_slices(files, num_slices=SLICES_PER_SUBJECT):
    files = sorted(files)
    mid = len(files) // 2
    half = num_slices // 2
    start = max(0, mid - half)
    end = start + num_slices
    if end > len(files):
        end = len(files)
        start = end - num_slices
    return files[start:end]


# =========================
# 10) Dataset
# =========================
class SubjectSliceDataset(Dataset):
    def __init__(self, zip_path, subjects, subject_to_files, labels_dict, transform=None):
        self.zip_path = zip_path
        self.subjects = subjects
        self.subject_to_files = subject_to_files
        self.labels_dict = labels_dict
        self.transform = transform
        self.zip_ref = None

    def _get_zip(self):
        if self.zip_ref is None:
            self.zip_ref = zipfile.ZipFile(self.zip_path, "r")
        return self.zip_ref

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        zip_ref = self._get_zip()
        sid = self.subjects[idx]
        files = select_middle_slices(self.subject_to_files[sid], SLICES_PER_SUBJECT)

        slice_tensors = []

        for fpath in files:
            try:
                with zip_ref.open(fpath) as f:
                    dcm = pydicom.dcmread(f)
                    img = dcm.pixel_array.astype(np.float32)
            except Exception:
                continue

            # robust normalization per slice
            img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)
            img = (img - np.mean(img)) / (np.std(img) + 1e-8)
            img = np.clip(img, -3, 3)
            img = (img - img.min()) / (img.max() - img.min() + 1e-8)

            img = Image.fromarray((img * 255).astype(np.uint8)).convert("RGB")

            if self.transform:
                img = self.transform(img)

            slice_tensors.append(img)

        # if some slices failed, pad by repeating last good slice
        if len(slice_tensors) == 0:
            # fallback blank image tensor
            blank = torch.zeros(3, IMG_SIZE, IMG_SIZE)
            slice_tensors = [blank for _ in range(SLICES_PER_SUBJECT)]
        elif len(slice_tensors) < SLICES_PER_SUBJECT:
            while len(slice_tensors) < SLICES_PER_SUBJECT:
                slice_tensors.append(slice_tensors[-1].clone())

        x = torch.stack(slice_tensors, dim=0)  # [S, 3, H, W]
        y = self.labels_dict[sid]

        return x, y


# =========================
# 11) Transforms
# =========================
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])


# =========================
# 12) Loaders
# =========================
train_dataset = SubjectSliceDataset(
    zip_path=zip_path,
    subjects=train_subjects,
    subject_to_files=subject_to_files,
    labels_dict=labels_dict,
    transform=train_transform
)

val_dataset = SubjectSliceDataset(
    zip_path=zip_path,
    subjects=val_subjects,
    subject_to_files=subject_to_files,
    labels_dict=labels_dict,
    transform=val_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)


# =========================
# 13) Class weights
# =========================
train_labels = [labels_dict[sid] for sid in train_subjects]
counts = Counter(train_labels)
print("Train subject class distribution:", counts)

total = sum(counts.values())
class_weights = torch.tensor(
    [total / counts[i] for i in range(3)],
    dtype=torch.float32,
    device=device
)


# =========================
# 14) Model
# =========================
class SubjectResNetAggregator(nn.Module):
    def __init__(self, num_classes=3):
        super().__init__()

        backbone = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        # keep convolutional feature extractor only
        self.feature_extractor = nn.Sequential(*list(backbone.children())[:-1])  # [B, 512, 1, 1]

        # freeze earlier layers, fine-tune layer4 and classifier
        for name, param in self.feature_extractor.named_parameters():
            param.requires_grad = False
            if "7." in name:  # layer4 block in Sequential indexing
                param.requires_grad = True

        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x: [B, S, 3, H, W]
        b, s, c, h, w = x.shape
        x = x.view(b * s, c, h, w)

        feats = self.feature_extractor(x)       # [B*S, 512, 1, 1]
        feats = feats.view(b, s, 512)           # [B, S, 512]

        # aggregate slice features at subject level
        feats = feats.mean(dim=1)               # [B, 512]

        feats = self.dropout(feats)
        out = self.classifier(feats)
        return out


model = SubjectResNetAggregator(num_classes=3).to(device)


# =========================
# 15) Training setup
# =========================
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=1
)


# =========================
# 16) Evaluation helpers
# =========================
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(imgs)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * labels.size(0)

            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())

    avg_loss = total_loss / total
    acc = correct / total

    return avg_loss, acc, all_preds, all_labels


# =========================
# 17) Training loop
# =========================
best_acc = 0.0
best_epoch = 0
epochs_without_improvement = 0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    seen = 0

    for imgs, labels in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        seen += labels.size(0)

    train_loss = running_loss / seen
    val_loss, val_acc, _, _ = evaluate(model, val_loader)

    scheduler.step(val_acc)

    print(f"\nEpoch {epoch+1}")
    print("Train Loss:", train_loss)
    print("Val Loss:", val_loss)
    print("Val Accuracy:", val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("\nEarly stopping triggered.")
        break

print("\nBest Val Accuracy:", best_acc)
print("Best Epoch:", best_epoch)
print("Best model saved to:", best_model_path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Total labeled subjects: 521
Total DICOM files: 1604356
Total subjects with images: 486
Subjects used: 450
Train subjects: 358
Val subjects: 92
Train subject class distribution: Counter({2: 204, 1: 89, 0: 65})

Epoch 1
Train Loss: 1.1906353854600278
Val Loss: 1.2775074554526287
Val Accuracy: 0.4673913043478261

Epoch 2
Train Loss: 1.0862470299172002
Val Loss: 1.2891157036242278
Val Accuracy: 0.3804347826086957

Epoch 3
Train Loss: 1.0083741931941923
Val Loss: 1.2239353086637415
Val Accuracy: 0.2717391304347826

Epoch 4
Train Loss: 0.8700360915514344
Val Loss: 1.16239943711654
Val Accuracy: 0.43478260869565216

Early stopping triggered.

Best Val Accuracy: 0.4673913043478261
Best Epoch: 1
Best model saved to: /content/drive/MyDrive/best_subject_resnet18_agg.pth


### Model 4: Improved ResNet Model
*Highest Accuracy at 0.565 / 5 Epochs*

In [ ]:
# =========================
# 1) Setup
# =========================
!pip install pydicom scikit-learn joblib

import os
import zipfile
import random
import numpy as np
import pandas as pd
import pydicom
import torch
import torch.nn as nn
from torchvision import transforms, models
from PIL import Image
from collections import Counter
from google.colab import drive

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import joblib

drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =========================
# CONFIG
# =========================
SEED = 42
IMG_SIZE = 224
SLICES_PER_SUBJECT = 15
CENTRAL_FRACTION = 0.60
MAX_SUBJECTS = None          # use all available subjects; set to e.g. 450 if you want faster runtime
FEATURE_BATCH_SIZE = 16      # for ResNet feature extraction
ARTIFACT_DIR = "/content/drive/MyDrive/adni_subject_feature_model"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.makedirs(ARTIFACT_DIR, exist_ok=True)


# =========================
# 2) Paths
# =========================
zip_path = "/content/drive/MyDrive/ADNI_DATA/images/MRI.zip"
diagnosis_csv_path = "Diagnositic_Summary.csv"


# =========================
# 3) Load Diagnosis CSV
# =========================
df = pd.read_csv(diagnosis_csv_path)

df = df[["PTID", "EXAMDATE", "DXNORM", "DXMCI", "DXAD"]].copy()
df["EXAMDATE"] = pd.to_datetime(df["EXAMDATE"], errors="coerce")

df = df.sort_values("EXAMDATE")
latest_df = df.groupby("PTID").tail(1)

labels_dict = {}

for _, row in latest_df.iterrows():
    if row["DXAD"] == 1:
        labels_dict[row["PTID"]] = 2
    elif row["DXMCI"] == 1:
        labels_dict[row["PTID"]] = 1
    elif row["DXNORM"] == 1:
        labels_dict[row["PTID"]] = 0

print("Total labeled subjects:", len(labels_dict))


# =========================
# 4) Open ZIP
# =========================
zip_ref = zipfile.ZipFile(zip_path, "r")
all_files = [f for f in zip_ref.namelist() if f.endswith(".dcm")]
zip_ref.close()

print("Total DICOM files:", len(all_files))


# =========================
# 5) Extract Subject ID
# =========================
def extract_subject_id(path):
    for part in path.split("/"):
        if "_S_" in part:
            return part
    return None


# =========================
# 6) Group by SUBJECT
# =========================
subject_to_files = {}

for f in all_files:
    subject_id = extract_subject_id(f)
    if subject_id in labels_dict:
        subject_to_files.setdefault(subject_id, []).append(f)

for sid in subject_to_files:
    subject_to_files[sid] = sorted(subject_to_files[sid])

print("Total subjects with images:", len(subject_to_files))


# =========================
# 7) Keep subjects with enough slices
# =========================
valid_subjects = [sid for sid, files in subject_to_files.items() if len(files) >= SLICES_PER_SUBJECT]

if MAX_SUBJECTS is not None:
    random.shuffle(valid_subjects)
    valid_subjects = valid_subjects[:MAX_SUBJECTS]

print("Subjects used:", len(valid_subjects))


# =========================
# 8) Stratified subject split
# =========================
subjects_by_class = {0: [], 1: [], 2: []}
for sid in valid_subjects:
    subjects_by_class[labels_dict[sid]].append(sid)

for cls in subjects_by_class:
    random.shuffle(subjects_by_class[cls])

train_subjects = []
val_subjects = []

for cls in [0, 1, 2]:
    cls_subjects = subjects_by_class[cls]
    split_idx = int(0.8 * len(cls_subjects))
    train_subjects.extend(cls_subjects[:split_idx])
    val_subjects.extend(cls_subjects[split_idx:])

random.shuffle(train_subjects)
random.shuffle(val_subjects)

print("Train subjects:", len(train_subjects))
print("Val subjects:", len(val_subjects))
print("Train subject class distribution:", Counter([labels_dict[s] for s in train_subjects]))
print("Val subject class distribution:", Counter([labels_dict[s] for s in val_subjects]))


# =========================
# 9) Slice Selection
# =========================
def select_central_slices(files, num_slices=SLICES_PER_SUBJECT, central_fraction=CENTRAL_FRACTION):
    files = sorted(files)

    n = len(files)
    start = int((1 - central_fraction) / 2 * n)
    end = int((1 + central_fraction) / 2 * n)

    central_files = files[start:end]
    if len(central_files) < num_slices:
        central_files = files

    if len(central_files) == num_slices:
        return central_files

    idxs = np.linspace(0, len(central_files) - 1, num_slices).astype(int)
    return [central_files[i] for i in idxs]


# =========================
# 10) DICOM preprocessing
# =========================
def load_dicom_as_pil(zip_ref, file_path):
    with zip_ref.open(file_path) as f:
        dcm = pydicom.dcmread(f, force=True)
        img = dcm.pixel_array.astype(np.float32)

    img = np.nan_to_num(img, nan=0.0, posinf=0.0, neginf=0.0)

    # percentile clipping helps more than raw min-max on medical scans
    p1, p99 = np.percentile(img, (1, 99))
    if p99 <= p1:
        p1, p99 = float(img.min()), float(img.max())

    img = np.clip(img, p1, p99)
    img = (img - p1) / (p99 - p1 + 1e-8)

    pil_img = Image.fromarray((img * 255).astype(np.uint8)).convert("RGB")
    return pil_img


# =========================
# 11) ResNet feature extractor
# =========================
weights = models.ResNet18_Weights.DEFAULT
resnet = models.resnet18(weights=weights)
feature_extractor = nn.Sequential(*list(resnet.children())[:-1]).to(device)
feature_extractor.eval()

for param in feature_extractor.parameters():
    param.requires_grad = False

preprocess = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


# =========================
# 12) Subject feature extraction
# =========================
def extract_subject_feature(subject_id):
    files = subject_to_files[subject_id]
    selected_files = select_central_slices(files)

    feats = []

    with zipfile.ZipFile(zip_path, "r") as zf:
        batch_imgs = []

        for fpath in selected_files:
            try:
                img = load_dicom_as_pil(zf, fpath)
                img = preprocess(img)
                batch_imgs.append(img)
            except Exception:
                continue

        if len(batch_imgs) == 0:
            return None

        # process in mini-batches in case you increase slices later
        for i in range(0, len(batch_imgs), FEATURE_BATCH_SIZE):
            batch = torch.stack(batch_imgs[i:i + FEATURE_BATCH_SIZE]).to(device)

            with torch.no_grad():
                out = feature_extractor(batch)   # [B, 512, 1, 1]
                out = out.squeeze(-1).squeeze(-1).cpu().numpy()  # [B, 512]

            feats.append(out)

    feats = np.concatenate(feats, axis=0)  # [num_slices, 512]

    # subject-level pooling
    feat_mean = feats.mean(axis=0)
    feat_std = feats.std(axis=0)
    feat_max = feats.max(axis=0)

    # final subject vector
    subject_feature = np.concatenate([feat_mean, feat_std, feat_max], axis=0)  # [1536]
    return subject_feature.astype(np.float32)


# =========================
# 13) Build train/val feature matrices
# =========================
def build_feature_matrix(subject_list, split_name="train"):
    X = []
    y = []
    kept_subjects = []

    for i, sid in enumerate(subject_list):
        feat = extract_subject_feature(sid)
        if feat is None:
            continue

        X.append(feat)
        y.append(labels_dict[sid])
        kept_subjects.append(sid)

        if (i + 1) % 25 == 0 or (i + 1) == len(subject_list):
            print(f"{split_name}: processed {i+1}/{len(subject_list)} subjects")

    X = np.vstack(X)
    y = np.array(y)

    return X, y, kept_subjects


X_train, y_train, kept_train_subjects = build_feature_matrix(train_subjects, "train")
X_val, y_val, kept_val_subjects = build_feature_matrix(val_subjects, "val")

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("y_train distribution:", Counter(y_train))
print("y_val distribution:", Counter(y_val))

np.save(os.path.join(ARTIFACT_DIR, "X_train.npy"), X_train)
np.save(os.path.join(ARTIFACT_DIR, "y_train.npy"), y_train)
np.save(os.path.join(ARTIFACT_DIR, "X_val.npy"), X_val)
np.save(os.path.join(ARTIFACT_DIR, "y_val.npy"), y_val)


# =========================
# 14) Candidate classifiers
# =========================
candidate_models = {
    "logreg_c0.1": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=0.1,
            max_iter=5000,
            class_weight="balanced",
            random_state=SEED
        ))
    ]),
    "logreg_c1.0": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=1.0,
            max_iter=5000,
            class_weight="balanced",
            random_state=SEED
        ))
    ]),
    "logreg_c3.0": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C=3.0,
            max_iter=5000,
            class_weight="balanced",
            random_state=SEED
        ))
    ]),
    "linearsvc_c0.1": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(
            C=0.1,
            class_weight="balanced",
            random_state=SEED,
            max_iter=5000
        ))
    ]),
    "linearsvc_c1.0": Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=SEED,
            max_iter=5000
        ))
    ]),
    "extratrees": ExtraTreesClassifier(
        n_estimators=500,
        class_weight="balanced",
        random_state=SEED,
        n_jobs=-1
    ),
}


# =========================
# 15) Train and compare
# =========================
results = []
best_name = None
best_model = None
best_acc = -1
best_macro_f1 = -1

for name, model in candidate_models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    acc = accuracy_score(y_val, y_pred)
    macro_f1 = f1_score(y_val, y_pred, average="macro")

    results.append((name, acc, macro_f1))
    print(f"\n{name}")
    print("Val Accuracy:", acc)
    print("Val Macro F1:", macro_f1)

    # prefer accuracy first, then macro f1 as tie-breaker
    if (acc > best_acc) or (acc == best_acc and macro_f1 > best_macro_f1):
        best_acc = acc
        best_macro_f1 = macro_f1
        best_name = name
        best_model = model

print("\nModel comparison:")
for name, acc, macro_f1 in results:
    print(f"{name}: acc={acc:.4f}, macro_f1={macro_f1:.4f}")

print("\nBest model:", best_name)
print("Best Val Accuracy:", best_acc)
print("Best Val Macro F1:", best_macro_f1)


# =========================
# 16) Final evaluation
# =========================
y_pred = best_model.predict(X_val)

print("\nClassification Report:")
print(classification_report(y_val, y_pred, digits=4))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_pred))


# =========================
# 17) Save artifacts
# =========================
joblib.dump(best_model, os.path.join(ARTIFACT_DIR, "best_subject_classifier.joblib"))

metadata = {
    "img_size": IMG_SIZE,
    "slices_per_subject": SLICES_PER_SUBJECT,
    "central_fraction": CENTRAL_FRACTION,
    "train_subjects": kept_train_subjects,
    "val_subjects": kept_val_subjects,
    "best_model_name": best_name,
    "best_val_accuracy": float(best_acc),
    "best_val_macro_f1": float(best_macro_f1),
}

joblib.dump(metadata, os.path.join(ARTIFACT_DIR, "metadata.joblib"))

print("\nArtifacts saved to:", ARTIFACT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda
Total labeled subjects: 521
Total DICOM files: 1604356
Total subjects with images: 486
Subjects used: 486
Train subjects: 387
Val subjects: 99
Train subject class distribution: Counter({2: 222, 1: 93, 0: 72})
Val subject class distribution: Counter({2: 56, 1: 24, 0: 19})
train: processed 25/387 subjects
train: processed 50/387 subjects
train: processed 75/387 subjects
train: processed 100/387 subjects
train: processed 125/387 subjects
train: processed 150/387 subjects
train: processed 175/387 subjects
train: processed 200/387 subjects
train: processed 225/387 subjects
train: processed 250/387 subjects
train: processed 275/387 subjects
train: processed 300/387 subjects
train: processed 325/387 subjects
train: processed 350/387 subjects
train: processed 375/387 subjects
train: processed 387/387 subjects
val: processed 25/99 subjects
val: proc

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



Artifacts saved to: /content/drive/MyDrive/adni_subject_feature_model
